In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from numpy import NaN
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import numpy as np
import pandas as pd

from sklearn import preprocessing
from sklearn.model_selection import train_test_split

2023-02-10 12:40:11.353557: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-02-10 12:40:11.468645: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-02-10 12:40:11.983668: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2023-02-10 12:40:11.983717: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] 

In [2]:
url = 'IoT Network Intrusion Dataset.csv'
data=pd.read_csv(url)
data.shape

(625783, 86)

In [3]:
# remove attribute 'difficulty_level'
data.drop(['Cat','Sub_Cat'],axis=1,inplace=True)
data.shape

(625783, 84)

In [4]:
# number of attack labels 
data['Label'].value_counts()

Anomaly    585710
Normal      40073
Name: Label, dtype: int64

In [5]:
dataset = data['Label']


In [6]:
# Checking if there are any NULL values in the dataset.

data.isnull().values.any()

False

In [7]:
# Checking which column/s contain NULL values.

[col for col in data if data[col].isnull().values.any()]

[]

In [8]:
from sklearn import preprocessing
from sklearn.preprocessing import OrdinalEncoder

ord_enc = OrdinalEncoder()
data["Flow_ID"] = ord_enc.fit_transform(data[["Flow_ID"]])
data["Src_IP"] = ord_enc.fit_transform(data[["Src_IP"]])
data["Dst_IP"] = ord_enc.fit_transform(data[["Dst_IP"]])
data["Timestamp"] = ord_enc.fit_transform(data[["Timestamp"]])

In [9]:
# Drop the repeated rows
df=data.drop_duplicates()

In [10]:
df.head(5)

,Flow_ID,Src_IP,Src_Port,Dst_IP,Dst_Port,Protocol,Timestamp,Flow_Duration,Tot_Fwd_Pkts,Tot_Bwd_Pkts,...,Fwd_Seg_Size_Min,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label
0,12446.0,25883.0,10000,203.0,10101,17,3496.0,75,1,1,...,0,0.0,0.0,0.0,0.0,75.0,0.000000,75.0,75.0,Anomaly
1,22760.0,34617.0,2179,200.0,554,6,3664.0,5310,1,2,...,0,0.0,0.0,0.0,0.0,2655.0,2261.327486,4254.0,1056.0,Anomaly
2,12691.0,25886.0,52727,200.0,9020,6,2082.0,141,0,3,...,0,0.0,0.0,0.0,0.0,70.5,0.707107,71.0,70.0,Anomaly
3,12704.0,25886.0,52964,200.0,9020,6,791.0,151,0,2,...,0,0.0,0.0,0.0,0.0,151.0,0.000000,151.0,151.0,Anomaly
4,611.0,25881.0,36763,317.0,1900,17,1040.0,153,2,1,...,0,0.0,0.0,0.0,0.0,76.5,0.707107,77.0,76.0,Anomaly


In [11]:
labl = df['Label']
dataset = df.loc[:, df.columns != 'Label'].astype('float64')

In [12]:
# Checking if all values are finite.

np.all(np.isfinite(dataset))

False

In [13]:
# Checking what column/s contain non-finite values.

nonfinite = [col for col in dataset if not np.all(np.isfinite(dataset[col]))]

nonfinite

['Flow_Byts/s', 'Flow_Pkts/s']

In [14]:
# Checking how many non-finite values each column contains.

finite = np.isfinite(dataset['Flow_Byts/s']).sum()

dataset.shape[0] - finite

323

In [15]:
# Checking how many non-finite values each column contains.

finite = np.isfinite(dataset['Flow_Pkts/s']).sum()

dataset.shape[0] - finite

323

In [16]:
# Same as before, since there is a small number of non-finite values we can safely remove them from the dataset
# without spoiling the dataset.

# Replacing infinite values with NaN values.
dataset = dataset.replace([np.inf, -np.inf], np.nan)

In [17]:
# We can see that now we have Nan values again.

np.any(np.isnan(dataset))

True

In [18]:
# Bringing the Labels back into the dataset before deliting Nan rows.

dataset = dataset.merge(labl, how='outer', left_index=True, right_index=True)

In [19]:
# Removing new NaN values.

dataset.dropna(inplace=True)

In [20]:
dataset.shape

(411382, 84)

In [21]:
dataset['Label'].value_counts()

Anomaly    372784
Normal      38598
Name: Label, dtype: int64

In [22]:
labels = dataset['Label']
features = dataset.loc[:, dataset.columns != 'Label'].astype('float64')

In [23]:
# For scaling the data, we use RobustScaler class from sklearn.

from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
scaler.fit(features)

features = scaler.transform(features)

In [24]:
from sklearn.preprocessing import LabelEncoder

LE = LabelEncoder()

LE.fit(labels)
labels = LE.transform(labels)

In [25]:
np.unique(labels)

array([0, 1])

In [26]:
# binary classification ##
 #changing attack category

bin_label = pd.DataFrame(dataset.Label.map(lambda x:'Normal' if x=='Normal' else 'ATTACK'))

In [27]:
bin_data = dataset.copy()
bin_data['Label'] = bin_label

In [28]:
LE1 = LabelEncoder()

enc_label = bin_label.apply(LE1.fit_transform)
bin_data['intrusion']= enc_label

In [29]:
LE1.classes_

array(['ATTACK', 'Normal'], dtype=object)

In [30]:
# one-hot-encoding for attack label

bin_data = pd.get_dummies(bin_data,columns=['Label'],prefix="",prefix_sep="")
bin_data['Label']= bin_label
bin_data

,Flow_ID,Src_IP,Src_Port,Dst_IP,Dst_Port,Protocol,Timestamp,Flow_Duration,Tot_Fwd_Pkts,Tot_Bwd_Pkts,...,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,intrusion,ATTACK,Normal,Label
0,12446.0,25883.0,10000.0,203.0,10101.0,17.0,3496.0,75.0,1.0,1.0,...,0.0,0.0,75.0,0.000000,75.0,75.0,0,1,0,ATTACK
1,22760.0,34617.0,2179.0,200.0,554.0,6.0,3664.0,5310.0,1.0,2.0,...,0.0,0.0,2655.0,2261.327486,4254.0,1056.0,0,1,0,ATTACK
2,12691.0,25886.0,52727.0,200.0,9020.0,6.0,2082.0,141.0,0.0,3.0,...,0.0,0.0,70.5,0.707107,71.0,70.0,0,1,0,ATTACK
3,12704.0,25886.0,52964.0,200.0,9020.0,6.0,791.0,151.0,0.0,2.0,...,0.0,0.0,151.0,0.000000,151.0,151.0,0,1,0,ATTACK
4,611.0,25881.0,36763.0,317.0,1900.0,17.0,1040.0,153.0,2.0,1.0,...,0.0,0.0,76.5,0.707107,77.0,76.0,0,1,0,ATTACK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
625773,62439.0,25889.0,60165.0,233.0,8899.0,17.0,3245.0,29.0,5.0,1.0,...,0.0,0.0,5.8,3.346640,11.0,3.0,0,1,0,ATTACK
625776,58871.0,21034.0,8739.0,205.0,19604.0,6.0,535.0,1092.0,0.0,2.0,...,0.0,0.0,1092.0,0.000000,1092.0,1092.0,0,1,0,ATTACK
625778,62081.0,25889.0,56112.0,233.0,8043.0,17.0,3443.0,277.0,1.0,1.0,...,0.0,0.0,277.0,0.000000,277.0,277.0,0,1,0,ATTACK
625779,18760.0,30623.0,4570.0,200.0,554.0,6.0,3637.0,1658.0,0.0,2.0,...,0.0,0.0,1658.0,0.000000,1658.0,1658.0,0,1,0,ATTACK


In [31]:
# creating dataframe with only numeric attributes of binary class and encoded label attribute
numeric_col = dataset.select_dtypes(include = 'number').columns

numeric_bin = bin_data[numeric_col]
numeric_bin['intrusion'] = bin_data['intrusion']

/tmp/ipykernel_15796/1352549749.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  numeric_bin['intrusion'] = bin_data['intrusion']


In [32]:
# extracting features which have more than 0.5 correlation (pearson correlation coefficient)

corr=numeric_bin.corr()
corr_y = abs(corr['intrusion'])
highest_corr = corr_y[corr_y >0.2]
highest_corr.sort_values(ascending=True)

Bwd_Pkt_Len_Max     0.200683
Pkt_Len_Max         0.204781
Bwd_Pkt_Len_Min     0.214931
Bwd_Pkt_Len_Mean    0.215361
Bwd_Seg_Size_Avg    0.215361
Protocol            0.221809
TotLen_Fwd_Pkts     0.240765
Subflow_Fwd_Byts    0.240765
Fwd_Pkt_Len_Mean    0.264598
Fwd_Seg_Size_Avg    0.264598
Src_Port            0.283235
ACK_Flag_Cnt        0.313216
Fwd_Pkt_Len_Max     0.315905
Fwd_Pkt_Len_Std     0.371807
Dst_Port            0.511449
intrusion           1.000000
Name: intrusion, dtype: float64

In [33]:
numeric_bin = bin_data[['Bwd_Pkt_Len_Max','Pkt_Len_Max','Bwd_Pkt_Len_Min','Bwd_Pkt_Len_Mean','Bwd_Seg_Size_Avg',
                        'Protocol','TotLen_Fwd_Pkts','Subflow_Fwd_Byts','Fwd_Pkt_Len_Mean','Fwd_Seg_Size_Avg',
                        'Src_Port','ACK_Flag_Cnt','Fwd_Pkt_Len_Max','Fwd_Pkt_Len_Std','Dst_Port']]

In [34]:
bin_data = numeric_bin.join(bin_data[['intrusion','Normal','ATTACK','Label']])

In [35]:
bin_data['intrusion'].value_counts()

0    372784
1     38598
Name: intrusion, dtype: int64

In [36]:
# classifier for binary classification dataset

import warnings
warnings.filterwarnings("ignore")


X = bin_data.iloc[:,0:15].to_numpy() # dataset excluding target attribute (encoded, one-hot-encoded,original)
y = bin_data['intrusion'] # target attribute

X_train, X_validation, Y_train, Y_validation = train_test_split(X, y, test_size=0.20, random_state=1)


In [37]:
X_train = preprocessing.scale(X_train)
X_train = preprocessing.normalize(X_train)

In [38]:
X_validation = preprocessing.scale(X_validation)
X_validation = preprocessing.normalize(X_validation)

In [39]:
print(len(X_train), "Training sequences",X_train.shape)
print(len(X_validation), "Validation sequences",X_validation.shape)

329105 Training sequences (329105, 15)
82277 Validation sequences (82277, 15)


In [40]:
X_train = np.reshape(X_train,(X_train.shape[0],X_train.shape[1],1))
X_validation = np.reshape(X_validation,(X_validation.shape[0],X_validation.shape[1],1))

In [41]:
X_train.shape

(329105, 15, 1)

In [42]:
# Using tanh and sigmoid as activation functions
import time

Model = keras.Sequential([

        keras.layers.Conv2D(96,(4,4),input_shape=(X_train.shape[1],X_train.shape[2],1),activation='relu',padding='same'),
        keras.layers.Conv2D(64,(3,3),activation="relu",padding='same'),
        keras.layers.Conv2D(32,(2,2),activation="relu",padding='same'),
        keras.layers.Flatten(),
        keras.layers.Dense(512,activation="relu"),
        keras.layers.Dense(128,activation="relu"),
        keras.layers.Dense(32,activation="relu"),
        keras.layers.Dense(2,activation="softmax"),
    
    
    ])

Model.compile(optimizer='adam',loss='sparse_categorical_crossentropy', metrics=['sparse_categorical_accuracy'])
start_time = time.time()
#Training the model
Model.fit(X_train, Y_train, epochs=5, batch_size=64) 
Model.summary()

# Final evaluation of the model
scores = Model.evaluate(X_validation, Y_validation, verbose=0)
delta = time.time()- start_time
print("Accuracy: %.2f%%" % (scores[1]*100))
print("Training time: %.2f sec" % (delta))

Epoch 1/5


2023-02-10 12:40:27.192409: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-02-10 12:40:28.370880: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14769 MB memory:  -> device: 0, name: Quadro RTX 5000, pci bus id: 0000:19:00.0, compute capability: 7.5
2023-02-10 12:40:28.371521: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 14769 MB memory:  -> device: 1, name: Quadro RTX 5000, pci bus id: 0000:1a:00.0, compute capability: 7.5
2023-02-10 12:40:28.371882: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/tas

5143/5143 [==============================] - 47s 9ms/step - loss: 0.0754 - sparse_categorical_accuracy: 0.9726
Epoch 2/5
5143/5143 [==============================] - 44s 9ms/step - loss: 0.0401 - sparse_categorical_accuracy: 0.9870
Epoch 3/5
5143/5143 [==============================] - 44s 9ms/step - loss: 0.0371 - sparse_categorical_accuracy: 0.9877
Epoch 4/5
5143/5143 [==============================] - 44s 9ms/step - loss: 0.0303 - sparse_categorical_accuracy: 0.9901
Epoch 5/5
5143/5143 [==============================] - 44s 8ms/step - loss: 0.0292 - sparse_categorical_accuracy: 0.9904
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 15, 1, 96)         1632      
                                                                 
 conv2d_1 (Conv2D)           (None, 15, 1, 64)         55360     
                                                       

In [43]:
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error
                             ,mean_absolute_error,roc_auc_score,confusion_matrix)
from sklearn import metrics

In [44]:
y_pred = Model.predict(X_validation, verbose = 0)
# predict crisp classes for test set
yhat_classes = np.argmax(y_pred,axis=1)
# reduce to 1d array
yhat_probs = y_pred[:, 0]

 
# accuracy: (tp + tn) / (p + n)
accuracy = accuracy_score(Y_validation, yhat_classes)
print('Accuracy: %f' % accuracy)
# precision tp / (tp + fp)
precision = precision_score(Y_validation, yhat_classes)
print('Precision: %f' % precision)
# recall: tp / (tp + fn)
recall = recall_score(Y_validation, yhat_classes)
print('Recall: %f' % recall)
# f1: 2 tp / (2 tp + fp + fn)
f1 = f1_score(Y_validation, yhat_classes)
print('F1 score: %f' % f1)


# ROC AUC
auc = roc_auc_score(Y_validation, yhat_probs)
print('ROC AUC: %f' % auc)
# confusion matrix
matrix = confusion_matrix(Y_validation, yhat_classes)
print(matrix)
# false alaram rate
false_alaram_rate = matrix[1,0]/(matrix[1,0]+matrix[0,0])
print('false alaram rate: %f' % false_alaram_rate)

Accuracy: 0.992088
Precision: 0.969761
Recall: 0.946005
F1 score: 0.957736
ROC AUC: 0.001703
[[74250   230]
 [  421  7376]]
false alaram rate: 0.005638
